# Exploratory 900-fit extension
The six nominal configurations map to D1–D4. Shared effective designs retain both nominal labels and reuse three seed values within each partition. All 50 partitions reuse the original cohort.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/"data/main_fit_metrics.csv").is_file())
OUT = Path(os.environ.get("ARTICLE_RESULTS_DIR", root/"generated")).resolve()
OUT.mkdir(parents=True, exist_ok=True)

def read(name):
    return pd.read_csv(root/"data"/(name+".csv"), float_precision="round_trip")

def save(name, frame):
    frame.to_csv(OUT/(name+".csv"), index=False)

def close(value, expected, tolerance):
    return abs(value-expected) <= tolerance

In [ ]:
metrics = read("extension_stage_metrics")
design = read("extension_design_and_sizes")
agreement = read("extension_decision_agreement")
fit = ["label", "effective_design", "split_idx", "model_seed_idx"]
assert len(metrics) == 5400 and len(design) == 900 and not design.duplicated(fit).any()
assert metrics.groupby(fit).size().eq(6).all()
assert metrics.split_idx.nunique() == 50
negative_f1 = 2*metrics.tn / (2*metrics.tn + metrics.fp + metrics.fn)
metrics["macro_f1"] = (metrics.f1 + negative_f1.fillna(0))/2
test = metrics[metrics.partition.eq("test")]
board = test[test.stage.eq("akida_forward")]
values = ["bacc", "auc", "precision", "recall", "specificity", "f1", "macro_f1"]
summary = board.groupby("effective_design")[values].mean()
summary.insert(0, "fits", board.groupby("effective_design").size())
assert summary.fits.tolist() == [150,300,150,300]
assert close(board.bacc.mean(), .5974, .00005)
assert close(board.auc.mean(), .6578, .00005)
assert close(board.f1.mean(), .4599, .00005)
assert close(board.macro_f1.mean(), .5494, .00005)
resources = design.groupby("effective_design").agg(params=("parameters", "median"), median_fbz=("fbz_bytes", "median"),
        min_fbz=("fbz_bytes", "min"), max_fbz=("fbz_bytes", "max"))
save("extension_design_summary", summary.join(resources).reset_index())
save("extension_label_summary", board.groupby("label")[values].mean().reset_index())
save("extension_stages", metrics.groupby(["stage", "partition"])[values].mean().reset_index())
save("extension_partition_means", board.groupby(["effective_design", "split_idx"])[values].mean().reset_index())

test_agreement = agreement[agreement.partition.eq("test")]
assert test_agreement.exact_classifications.sum() == 734 and test_agreement.decisions.sum() == 42300
share = 1 - test_agreement.different_decisions.sum() / test_agreement.decisions.sum()
assert close(share, .9871, .00005)
print(f"Software–board decision agreement: {100*share:.2f}%")
display(summary, resources)

In [ ]:
wide = test.pivot(index=fit, columns="stage", values=["bacc","auc"])
deltas = []
for a,b in [("float","akida_predict"), ("akida_predict","akida_forward")]:
    bacc_delta = wide.bacc[b]-wide.bacc[a]
    deltas.append({"comparison": f"{a} → {b}", "mean_bacc_delta": bacc_delta.mean(), "median_bacc_delta": bacc_delta.median(),
        "equal_bacc": bacc_delta.abs().lt(1e-12).sum(), "mean_auc_delta": (wide.auc[b]-wide.auc[a]).mean()})
save("extension_stage_deltas", pd.DataFrame(deltas))